# SatQuery AI: fine-tune the 4-band land-cover classifier on a larger BigEarthNet subset (GPU)

This runs the same scripts as the CPU run (`ml/landcover_patch`), with a bigger subset and more epochs:
`download_bigearthnet_subset.py` -> `train.py --device cuda` -> `eval.py`.

1. Runtime -> Change runtime type -> **GPU**.
2. Fill in `REPO_URL` (your fork / clone URL) and `BRANCH` below.
3. Run all cells. Download `resnet18_4band_ben.pt` at the end and copy it to `data/models/landcover_patch/` on your machine (or point `LANDCOVER_PATCH_PATH` at it).

Dataset: [timm/bigearthnet-v2-rgb-nir-swir](https://huggingface.co/datasets/timm/bigearthnet-v2-rgb-nir-swir) (CDLA-Permissive-1.0). Only RGB + NIR + labels are fetched, from the official geographic splits. Each 100k training patches is ~4.4 GB of download.

In [ ]:
REPO_URL = ""          # e.g. your GitHub clone URL; required
BRANCH = "develop"
TRAIN_PATCHES, VAL_PATCHES, TEST_PATCHES = 100_000, 10_000, 20_000
EPOCHS, BATCH_SIZE = 12, 128
assert REPO_URL, "Set REPO_URL first"

In [ ]:
!git clone --depth 1 --branch {BRANCH} {REPO_URL} satquery
%cd satquery
# Colab already has CUDA torch/torchvision; only add what the scripts need.
!pip -q install pyarrow huggingface_hub pyyaml python-dotenv
import torch; print(torch.__version__, torch.cuda.is_available())

In [ ]:
import yaml, pathlib
cfg = yaml.safe_load(open("ml/landcover_patch/config.yaml"))
cfg["dataset"]["patches"] = {"train": TRAIN_PATCHES, "validation": VAL_PATCHES, "test": TEST_PATCHES}
cfg["dataset"]["subset_dir"] = "data/bigearthnet_subset_colab"
cfg["output_dir"] = "data/models/landcover_patch_colab"
cfg["checkpoint"] = "data/models/landcover_patch_colab/resnet18_4band_ben.pt"
cfg["train"].update({"batch_size": BATCH_SIZE, "num_threads": 4, "num_workers": 2})
cfg["train"]["finetune"].update({"epochs": EPOCHS, "patience": 3})
pathlib.Path("colab_config.yaml").write_text(yaml.safe_dump(cfg))
print(yaml.safe_dump(cfg))

In [ ]:
!python ml/landcover_patch/download_bigearthnet_subset.py --config colab_config.yaml --dry-run

In [ ]:
!python ml/landcover_patch/download_bigearthnet_subset.py --config colab_config.yaml

In [ ]:
!python ml/landcover_patch/train.py --config colab_config.yaml --device cuda

In [ ]:
!python ml/landcover_patch/eval.py --config colab_config.yaml

In [ ]:
from google.colab import files
files.download("data/models/landcover_patch_colab/resnet18_4band_ben.pt")
files.download("data/models/landcover_patch_colab/eval_test.json")
files.download("data/models/landcover_patch_colab/train_log.json")